In [1]:
# build_bipartite_variants.py
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import random


In [2]:

# ---------------- CONFIG ----------------
NODES_CSV = "/mnt/data/nodes.csv"
EMBED_NPY = "/mnt/data/embeddings.npy"   # used for validation of node count
OUT_DIR = "/mnt/data/bipartite_variants"
os.makedirs(OUT_DIR, exist_ok=True)
RANDOM_SEED = 42
# percent variants (as decimals)
VARIANTS = {"40": 0.40, "50": 0.50, "60": 0.60}
# ----------------------------------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [3]:

# 1) load nodes
nodes = pd.read_csv(NODES_CSV)
N = len(nodes)
print(f"Loaded nodes: {N}")


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/nodes.csv'

In [ ]:

# 2) confirm embeddings shape if present
if os.path.exists(EMBED_NPY):
    emb = np.load(EMBED_NPY)
    assert emb.shape[0] == N, f"embeddings rows {emb.shape[0]} != nodes {N}"
    print("Embeddings shape:", emb.shape)

# 3) identify partitions
if "node_type" not in nodes.columns:
    raise SystemExit("nodes.csv must contain a 'node_type' column (0=email,1=reply).")

email_idx = nodes[nodes["node_type"] == 0].index.to_numpy()
reply_idx = nodes[nodes["node_type"] == 1].index.to_numpy()
n_email = len(email_idx)
n_reply = len(reply_idx)
print(f"Emails: {n_email}, Replies: {n_reply}")

# 4) compute maximum possible edges
max_edges = int(n_email) * int(n_reply)
print("Max possible bipartite edges (|U|*|V|):", max_edges)

# 5) For each target percent, sample edges uniformly from the bipartite space
for tag, frac in VARIANTS.items():
    target = int(round(max_edges * frac))
    print(f"\nBuilding variant {tag}% → target edges: {target}")

    edges_set = set()
    # sampling strategy: sample in blocks to reduce overhead
    block = 1000000  # number of candidate pairs to propose per loop (tune for speed/memory)
    # if target is small relative to max_edges we sample until we hit target
    # but ensure we don't infinite-loop: if target > max_edges throw
    if target > max_edges:
        raise ValueError("Target exceeds max edges")

    # Keep sampling random email indices and reply indices until we collect target unique edges
    # Represent edge as single integer key: e_idx * n_reply + r_idx (use position in partitions)
    # But we will store final edges as original node indices
    # Map email_pos->node_id and reply_pos->node_id
    email_pos_to_id = email_idx  # already the node index
    reply_pos_to_id = reply_idx

    # We'll sample by picking random positions (pos in email array, pos in reply array)
    # and deduplicating using set of pairs
    pbar = tqdm(total=target, desc=f"Sampling {tag}%")
    while len(edges_set) < target:
        # sample a batch of positions
        batch_size = min(100000, target - len(edges_set))
        e_pos = np.random.randint(0, n_email, size=batch_size)
        r_pos = np.random.randint(0, n_reply, size=batch_size)
        for ep, rp in zip(e_pos, r_pos):
            key = (int(ep), int(rp))
            if key not in edges_set:
                edges_set.add(key)
                pbar.update(1)
                if len(edges_set) >= target:
                    break
    pbar.close()

    # Convert sampled position pairs to original node ids
    src_nodes = []
    dst_nodes = []
    for (ep, rp) in edges_set:
        src_nodes.append(int(email_pos_to_id[ep]))
        dst_nodes.append(int(reply_pos_to_id[rp]))

    edge_index = np.vstack([src_nodes, dst_nodes]).astype(np.int64)
    # transpose to shape [2, E]
    edge_index = edge_index
    print(f"Variant {tag}%: final edges = {edge_index.shape[1] if edge_index.ndim==2 else edge_index.shape}")
    # save
    np.save(os.path.join(OUT_DIR, f"edge_index_{tag}.npy"), edge_index)
    print("Saved:", os.path.join(OUT_DIR, f"edge_index_{tag}.npy"))

print("\nAll variants created in", OUT_DIR)
